In [ ]:
!pip install langchain langgraph langchain-openai langchain-core langchain-community langchain-experimental fpdf pdfplumber

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 3.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.1/68.1 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.7/87.7 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 60.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 210.1/210.1 kB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 84.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 55.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 97.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 4.3 MB/s eta 0:00:00
  Created wheel for fpdf: filename=fpdf-1.7.2-py2.py3-none-any.whl size=40704 sha256=628e5242c4047

In [16]:
import os
import requests
import warnings
import random
from typing import Annotated, Literal
from typing_extensions import TypedDict
from pydantic import BaseModel, Field
from fpdf import FPDF
import pdfplumber

In [18]:
from langchain_core.runnables import RunnableConfig
from langchain_core.messages import AIMessage, HumanMessage
from langchain_core.prompts import PromptTemplate
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain_community.agent_toolkits import FileManagementToolkit
from langchain_experimental.tools.python.tool import PythonAstREPLTool
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode
from langgraph.checkpoint.memory import MemorySaver

from google.colab import drive
from google.colab import userdata

In [19]:
warnings.filterwarnings("ignore")

In [20]:
# --- [점검 및 설정] ---
def validate_setup():
    print("🔍 실행 환경을 점검 중입니다...")

    # 1. API 키 확인
    openai_key = userdata.get('OPENAI_KEY')
    tavily_key = userdata.get('TAVILY_API_KEY')

    if not openai_key:
        print("\n❌ 에러: 'OPENAI_KEY'가 설정되지 않았습니다.")
        print("👉 해결 방법: Colab 왼쪽 사이드바의 '열쇠(Secrets)' 아이콘을 클릭하여 'OPENAI_KEY'를 추가하고 '노트북 액세스 권한'을 활성화하세요.")
        return False

    if not tavily_key:
        print("\n❌ 에러: 'TAVILY_API_KEY'가 설정되지 않았습니다.")
        print("👉 해결 방법: '열쇠(Secrets)' 메뉴에서 'TAVILY_API_KEY'를 추가하고 액세스 권한을 부여하세요.")
        return False

    os.environ["OPENAI_API_KEY"] = openai_key
    os.environ["TAVILY_API_KEY"] = tavily_key

    # 2. 폰트 확인 안내
    font_path_drive = "/content/drive/MyDrive/fonts/NotoSansKR.ttf"
    if not os.path.exists(font_path_drive):
        print("\n⚠️ 안내: 구글 드라이브 내 폰트 경로(/content/drive/MyDrive/fonts/NotoSansKR.ttf)를 찾을 수 없습니다.")
        print("ℹ️ 드라이브가 마운트되지 않았거나 파일이 없는 경우, GitHub에서 자동으로 폰트를 다운로드하여 진행합니다.")

    print("✅ 모든 환경 설정이 확인되었습니다.\n")
    return True

In [21]:
# 점검 실행
if not validate_setup():
    raise SystemExit("환경 설정 오류로 인해 중단합니다.")

🔍 실행 환경을 점검 중입니다...

⚠️ 안내: 구글 드라이브 내 폰트 경로(/content/drive/MyDrive/fonts/NotoSansKR.ttf)를 찾을 수 없습니다.
ℹ️ 드라이브가 마운트되지 않았거나 파일이 없는 경우, GitHub에서 자동으로 폰트를 다운로드하여 진행합니다.
✅ 모든 환경 설정이 확인되었습니다.



In [22]:
# --- 클래스 및 도구 정의 ---

class State(TypedDict):
    query : Annotated[str, "User Question"]
    answer : Annotated[str, "LLM response"]
    messages : Annotated[list, add_messages]
    tool_call : Annotated[dict, "Tool Call Result"]

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

In [23]:
@tool
def read_pdf(file_path: str):
    """PDF 파일에서 텍스트를 추출하는 도구입니다."""
    try:
        text = ""
        with pdfplumber.open(file_path) as pdf:
            for page in pdf.pages:
                page_text = page.extract_text()
                if page_text:
                    text += page_text + "\n"
        return text.strip() if text.strip() else "❌ PDF에서 텍스트를 추출할 수 없습니다."
    except Exception as e:
        return f"❌ PDF 읽기 오류: {str(e)}"

In [24]:
@tool
def write_pdf(content: str, filename: str = "output.pdf", summary: bool = True):
    """텍스트를 PDF 파일로 저장하는 도구입니다."""

    if summary:
        # 전문가 스타일의 요약 프롬프트 적용
        job_summary_prompt = PromptTemplate.from_template("""
                당신은 **AI 분야 전문 채용 컨설턴트**입니다. 제공된 검색 결과(content)를 바탕으로 'AI 엔지니어 채용 현황 보고서'를 작성해야 합니다.

                **[작성 지침]**
                1. **구조화된 출력**: 보고서는 다음 항목을 반드시 포함해야 합니다.
                   - **I. 서론**: 현재 AI 채용 시장의 전반적인 트렌드 요약 (1-2문장)
                   - **II. 주요 기업별 채용 요약**: (검색된 기업 중 주요 기업 3-5곳 선정)
                     - **[기업명]**: 직무명, 핵심 요구 역량(기술 스택 포함), 우대 사항
                   - **III. 공통 요구 역량 및 핵심 기술**: 검색 결과를 종합하여 가장 많이 요구되는 기술 5가지와 그 이유
                   - **IV. 결론 및 인사이트**: 구직자를 위한 조언 및 향후 전망 (1-2문장)
                2. **가독성**: PDF로 저장될 것이므로, 명확한 제목과 글머리 기호(*, -)를 사용하여 읽기 쉽게 작성하세요.
                3. **한글 작성**: 모든 내용은 한글로 작성합니다.

                **[검색 결과 (Content)]**
                {content}
                """)

        chain = job_summary_prompt | llm
        content = chain.invoke({"content": content}).content

    # 한글 폰트 설정 및 자동 다운로드
    font_url = "https://github.com/google/fonts/raw/main/ofl/notosanskr/NotoSansKR%5Bwght%5D.ttf"
    font_dir = "./fonts/"
    font_path = font_dir + "NotoSansKR.ttf"

    if not os.path.exists(font_path):
        os.makedirs(font_dir, exist_ok=True)
        resp = requests.get(font_url)
        with open(font_path, "wb") as f:
            f.write(resp.content)

    pdf = FPDF()
    pdf.add_page()
    pdf.set_auto_page_break(auto=True, margin=15)

    try:
        pdf.add_font("NotoSans", "", font_path, uni=True)
        pdf.set_font("NotoSans", size=12)
    except:
        return "❌ 한글 폰트를 설정할 수 없습니다."

    for line in content.split("\\n"):
        pdf.multi_cell(0, 10, line)

    pdf.output(f"./{filename}")
    return f"📄 {filename} 저장 완료"

In [25]:
# 에이전트 구성
tools = [TavilySearchResults(max_results=10), PythonAstREPLTool(), write_pdf, read_pdf,
         *FileManagementToolkit(selected_tools=["file_delete","list_directory"]).get_tools()]
llm_with_tools = llm.bind_tools(tools)

In [26]:
# 그래프 로직 함수들
def shorterm_memory(state:State):
    return state["messages"][-8:-1] if len(state["messages"]) > 8 else state["messages"][:-1] if len(state["messages"]) > 1 else ""

class Checkers(BaseModel):
    yes_no : Literal["yes", "no"] = Field(..., description="History answerable?")
    end : Literal["end", "tool"] = Field(..., description="Solved?")

def history_check(state:State):
    checker = llm.with_structured_output(Checkers)
    res = checker.invoke(f"질문: {state['query']} 기록: {shorterm_memory(state)}")
    return res.yes_no

def memory_chat(state:State):
    res = llm.invoke(f"기록 참조 답변: {state['query']}")
    return {"answer": res.content, "messages": res}

def select(state: State):
    res = llm_with_tools.invoke(state["query"])
    return {"messages": res, "tool_call": res.tool_calls} if hasattr(res, "tool_calls") and res.tool_calls else {"messages": AIMessage(content="도구 미선택"), "tool_call": "없음"}

def answer_check(state:State):
    checker = llm.with_structured_output(Checkers)
    res = checker.invoke(f"해결됨? {state['answer']}")
    return res.end

In [27]:
# 그래프 빌드
graph_builder = StateGraph(State)
graph_builder.add_node("setup", lambda s: s)
graph_builder.add_node("memory_chat", memory_chat)
graph_builder.add_node("select", select)
graph_builder.add_node("tools", ToolNode(tools))
graph_builder.add_node("response", lambda s: {"answer": s["messages"][-1]})

graph_builder.add_edge(START, "setup")
graph_builder.add_conditional_edges("setup", history_check, {"yes":"memory_chat", "no":"select"})
graph_builder.add_edge("select", "tools")
graph_builder.add_edge("tools", "response")
graph_builder.add_edge("memory_chat", "response")
graph_builder.add_conditional_edges("response", answer_check, {"end":END, "tool":"select"})

graph = graph_builder.compile(checkpointer=MemorySaver())

def streaming(query, config):
    for step in graph.stream({"messages":("user", query), "query":query}, config=config, stream_mode="values"):
        if "messages" in step: step["messages"][-1].pretty_print()

In [30]:
# --- [최종 실행: 구인 공고 리포트 생성] ---

config = RunnableConfig(recursion_limit=25, configurable={"thread_id": random.randint(1,999999)})

recruitment_query = """
방금 검색했던 2026년 AI 엔지니어 채용 정보를 바탕으로 보고서를 작성해줘.
이미 검색은 완료되었으니, 수집된 정보를 바탕으로 'AI_Engineer_Job_Report_2026.pdf' 파일을 즉시 생성해줘.
한글 폰트(NotoSansKR)가 깨지지 않게 주의해서 작성해줘.
"""

print("⏳ AI 에이전트가 채용 공고를 검색하고 보고서를 생성 중입니다... (약 1-3분 소요)")
streaming(recruitment_query, config)
print("\\n✅ 보고서 생성이 완료되었습니다.")

⏳ AI 에이전트가 채용 공고를 검색하고 보고서를 생성 중입니다... (약 1-3분 소요)
================================ Human Message =================================


방금 검색했던 2026년 AI 엔지니어 채용 정보를 바탕으로 보고서를 작성해줘.
이미 검색은 완료되었으니, 수집된 정보를 바탕으로 'AI_Engineer_Job_Report_2026.pdf' 파일을 즉시 생성해줘.
한글 폰트(NotoSansKR)가 깨지지 않게 주의해서 작성해줘.

================================ Human Message =================================


방금 검색했던 2026년 AI 엔지니어 채용 정보를 바탕으로 보고서를 작성해줘.
이미 검색은 완료되었으니, 수집된 정보를 바탕으로 'AI_Engineer_Job_Report_2026.pdf' 파일을 즉시 생성해줘.
한글 폰트(NotoSansKR)가 깨지지 않게 주의해서 작성해줘.

================================== Ai Message ==================================
Tool Calls:
  write_pdf (call_1rEXLbPJv2n0Znrb2QfrPhGH)
 Call ID: call_1rEXLbPJv2n0Znrb2QfrPhGH
  Args:
    content: # 2026년 AI 엔지니어 채용 정보 보고서

## 1. 서론
2026년 AI 엔지니어 채용 시장은 급격한 변화와 성장을 겪고 있습니다. 인공지능 기술의 발전과 함께 다양한 산업에서 AI 엔지니어에 대한 수요가 증가하고 있습니다. 본 보고서는 2026년 AI 엔지니어 채용에 대한 주요 트렌드와 요구 사항을 분석합니다.

## 2. 채용 시장 동향
- **수요 증가**: AI 기술의 발전으로 인해 기업들은 AI 엔지니어를 적극적으로 채용하고 있습니다. 특히, 데이터 분석, 머신